# CUDA Particle Simulation — Colab Demo

Real-time GPU particle simulation running on a Tesla T4.  
100 particles on crossing H/V lines — elastic wall bounce + particle collision.  

**Runtime:** `Runtime → Change runtime type → T4 GPU`

In [ ]:
# Verify T4 GPU is attached
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader

In [ ]:
import os

REPO = '/content/cuda_particles'
if not os.path.exists(REPO):
    !git clone https://github.com/GaiBrutman/cuda_particles.git {REPO}
else:
    !git -C {REPO} pull --ff-only

%cd {REPO}

In [ ]:
# Build for T4 (SM 7.5 = Turing)
BUILD = f'{REPO}/build'
!cmake {REPO} -B {BUILD} \
    -DCMAKE_BUILD_TYPE=Release \
    -DCMAKE_CUDA_ARCHITECTURES=75 \
    -DCPU_ONLY=OFF
!cmake --build {BUILD} --parallel $(nproc)

In [ ]:
import subprocess, pathlib

FRAMES = '/tmp/frames'
pathlib.Path(FRAMES).mkdir(exist_ok=True)

result = subprocess.run([
    f'{BUILD}/particles',
    '--particles', '100',
    '--frames',    '300',
    '--width',     '1280',
    '--height',    '720',
    '--dt',        '0.016',
    '--output',    FRAMES,
], capture_output=True, text=True)

print(result.stdout)
if result.returncode != 0:
    print(result.stderr)

frames = sorted(pathlib.Path(FRAMES).glob('frame_*.png'))
print(f'Generated {len(frames)} frames')

In [ ]:
# ── Scale test: 500 K particles ───────────────────────────────────────────────
# Uncomment to benchmark large particle count on T4.

# import time
# start = time.perf_counter()
# subprocess.run([
#     f'{BUILD}/particles',
#     '--particles', '500000',
#     '--frames',    '60',
#     '--output',    '/tmp/frames_large',
# ], check=True)
# elapsed = time.perf_counter() - start
# print(f'500K × 60 frames in {elapsed:.1f}s  ({60/elapsed:.1f} sim-fps)')

In [ ]:
!pip install -q matplotlib pillow

In [ ]:
import glob
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from PIL import Image
from IPython.display import HTML

paths  = sorted(glob.glob(f'{FRAMES}/frame_*.png'))
frames_arr = [np.asarray(Image.open(p).convert('RGBA')) for p in paths]

h, w = frames_arr[0].shape[:2]
fig, ax = plt.subplots(figsize=(w / 100, h / 100), dpi=100)
fig.patch.set_facecolor('black')
ax.axis('off')
fig.tight_layout(pad=0)

im      = ax.imshow(frames_arr[0], origin='upper', aspect='equal',
                    extent=[0, w, h, 0], animated=True)
counter = ax.text(8, 14, 'frame 0000', color='gray',
                  fontsize=8, fontfamily='monospace', va='top')

def update(i):
    im.set_data(frames_arr[i])
    counter.set_text(f'frame {i:04d}')
    return im, counter

anim = animation.FuncAnimation(fig, update, frames=len(frames_arr),
                                interval=33, blit=True)
HTML(anim.to_jshtml())

In [ ]:
# Export as GIF (optional — takes ~30s for 300 frames)
GIF = '/tmp/particles.gif'
writer = animation.PillowWriter(fps=30)
anim.save(GIF, writer=writer)
print(f'Saved {GIF}')

from IPython.display import Image as IPyImage
IPyImage(GIF)